# PathoScan CNN training on Kaggle

This notebook trains an EfficientNet-B0 convolutional neural network on the ISIC 2019 skin-lesion dataset and launches a browser-accessible Gradio demo.

**Kaggle setup:** Add the ISIC 2019 dataset to the notebook, select a GPU accelerator, and enable Internet so ImageNet weights and Gradio can be installed. The notebook does not copy the full image archive; it creates symlinks in `/kaggle/working`.

In [ ]:
!pip install -q gradio
import os, csv, random, shutil
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
CLASS_NAMES = ('MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC', 'SCC')
ROOT = Path('/kaggle/input')
CSV_PATH = next(ROOT.rglob('ISIC_2019_Training_GroundTruth.csv'))
IMAGE_ROOT = next(p for p in ROOT.rglob('*') if p.is_dir() and any(p.glob('*.jpg')))
WORK_ROOT = Path('/kaggle/working/isic2019')
print('Labels:', CSV_PATH)
print('Images:', IMAGE_ROOT)

rows = []
with CSV_PATH.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        image_id = row['image'].strip()
        positive = [c for c in CLASS_NAMES if row.get(c, '').strip() == '1.0']
        image_path = IMAGE_ROOT / f'{image_id}.jpg'
        if len(positive) == 1 and image_path.exists():
            rows.append((image_path, positive[0]))
print('Usable labelled images:', len(rows))
assert len(rows) > 1000, 'Dataset was not discovered correctly.'

for c in CLASS_NAMES:
    (WORK_ROOT / c).mkdir(parents=True, exist_ok=True)
for image_path, label in rows:
    link = WORK_ROOT / label / image_path.name
    if not link.exists():
        link.symlink_to(image_path)
print('ImageFolder layout ready:', WORK_ROOT)

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.15),
    transforms.ToTensor(), transforms.Normalize((.485,.456,.406), (.229,.224,.225))
])
eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize((.485,.456,.406), (.229,.224,.225))
])

from torchvision.datasets import ImageFolder
full_train = ImageFolder(WORK_ROOT, transform=train_tfms)
full_eval = ImageFolder(WORK_ROOT, transform=eval_tfms)
indices = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED)).tolist()
val_n = int(len(indices) * .2)
val_idx, train_idx = indices[:val_n], indices[val_n:]
train_ds = torch.utils.data.Subset(full_train, train_idx)
val_ds = torch.utils.data.Subset(full_eval, val_idx)
train_labels = torch.tensor([full_train.targets[i] for i in train_idx])
counts = torch.bincount(train_labels, minlength=len(CLASS_NAMES)).float()
weights = (1 / counts.clamp_min(1)); weights /= weights.sum()
sampler = WeightedRandomSampler(weights[train_labels], len(train_labels), replacement=True)
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print('Classes:', full_train.classes)
print('Train/validation:', len(train_ds), len(val_ds))

In [ ]:
weights_enum = models.EfficientNet_B0_Weights.IMAGENET1K_V1
model = models.efficientnet_b0(weights=weights_enum)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES))
model.to(device)
criterion = nn.CrossEntropyLoss(weight=(1 / counts.clamp_min(1)).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
best_acc = -1
for epoch in range(10):
    model.train(); running = 0
    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels); loss.backward(); optimizer.step()
        running += loss.item() * labels.size(0)
    model.eval(); correct = total = 0
    with torch.inference_mode():
        for images, labels in val_loader:
            predictions = model(images.to(device, non_blocking=True)).argmax(1).cpu()
            correct += (predictions == labels).sum().item(); total += labels.numel()
    acc = correct / total; scheduler.step()
    print(f'Epoch {epoch+1}/10 | loss={running/len(train_ds):.4f} | val_accuracy={acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        torch.save({'model_state_dict': model.state_dict(), 'class_names': list(CLASS_NAMES), 'validation_accuracy': acc}, '/kaggle/working/pathoscan_efficientnet_b0.pt')
print('Best validation accuracy:', best_acc)

In [ ]:
import gradio as gr
checkpoint = torch.load('/kaggle/working/pathoscan_efficientnet_b0.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict']); model.eval()
labels = {
    'MEL': 'Melanoma', 'NV': 'Nevus', 'BCC': 'Basal Cell Carcinoma',
    'AKIEC': 'Actinic Keratosis', 'BKL': 'Benign Keratosis',
    'DF': 'Dermatofibroma', 'VASC': 'Vascular Lesion', 'SCC': 'Squamous Cell Carcinoma'
}
def predict(image):
    if image is None: return {'error': 'Upload an image.'}
    tensor = eval_tfms(Image.fromarray(image).convert('RGB')).unsqueeze(0).to(device)
    with torch.inference_mode():
        probs = torch.softmax(model(tensor), 1)[0].cpu()
    order = torch.argsort(probs, descending=True)[:3]
    result = {labels[CLASS_NAMES[i]]: round(float(probs[i]), 4) for i in order}
    top = CLASS_NAMES[order[0]]
    result['screening_note'] = 'AI screening only; consult a qualified healthcare professional.'
    result['top_prediction'] = f'{labels[top]} ({float(probs[order[0]])*100:.1f}%)'
    return result
demo = gr.Interface(fn=predict, inputs=gr.Image(type='numpy'), outputs=gr.JSON(), title='PathoScan CNN Skin Lesion Screening')
demo.launch(share=True)

The final cell prints a public Gradio URL. Open that URL in your browser and upload a lesion image. Download `pathoscan_efficientnet_b0.pt` from the Kaggle output to use the checkpoint in the local FastAPI application. This model is for research screening only and must not be used as a diagnosis.